                               #######             
                              #                    
        ############ -#####+.##     ######         |	DataDrivenConstruction.io
        ##+########## .-##+ #-   .+########        |	
        ####     ####.     ..    #      ###        |	Dive into the future of flexibility in processing data 
        ####     .#+#.     #    .# .               |	without using BIM tools!
        ####     -#+#.    #.     .#.               |	
        ####      #+#.   ##-                       |	
        ####     ####        #.  #     -####       |	
        ############  .###+ #. ############        |
        #########.   +#### #+.   ########          |	If you have any questions, concerns or special requests,
                           #+                      |	please contact us at info@datadrivenconstruction.io	
                          #                        
                         #  

To work with data in Jupyter Notebook, we first need to **load the project dataset** obtained during conversion.
The first cell converts a model (Revit, IFC or DWG) with the DDC converters and builds the flat table (`ddc table`)
with the columns of the old `*_rvt.xlsx`. By default it uses the open IFC sample of this repository; set `DDC_MODEL`
or the variable `model` to your own file.

In [ ]:
import os, sys, tempfile
from pathlib import Path

# Folder of this repository (it holds DDC_WINDOWS_Converters, ddc-tools and Sample_Projects)
REPO = Path(os.environ.get("DDC_REPO", r"C:\DDC\cad2data-Revit-IFC-DWG"))
sys.path.insert(0, str(REPO / "ddc-tools" / "src"))   # or install once:  pip install "<repo>\ddc-tools"
from ddc_tools import convert, table

# Input: a model file (.rvt, .ifc, .dwg) or the folder of a converter result.
# The default is the open IFC sample of this repository.
model = os.environ.get("DDC_MODEL", str(REPO / "Sample_Projects" / "Ifc2x3_Duplex_Architecture.ifc"))
OUT = Path(os.environ.get("DDC_OUT", Path(tempfile.gettempdir()) / "ddc_notebooks"))
OUT.mkdir(parents=True, exist_ok=True)

# Model file -> converter -> flat table with the columns of the old *_rvt.xlsx / *_ifc.xlsx
# (the same as the command line:  ddc convert <model> --xlsx <file>)
xlsx = str(OUT / "model_table.xlsx")
if os.path.isfile(model) and model.lower().endswith((".rvt", ".rfa", ".dwg")):
    convert.convert(model, work=str(OUT / (Path(model).stem + ".converted")), xlsx=xlsx)
else:                                    # an .ifc file or a converter output folder
    table.write_xlsx(table.build_table(model), xlsx)
print("table:", xlsx)

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
df = pd.read_excel(xlsx, sheet_name="Project_Data")

In [ ]:
df.head()  #Show the first 5 lines of the Dataframe

In [ ]:
# Write parameters into the variables
parameter_1 = 'Type Name'
parameter_2 = 'Volume'

#Summing the entire project for one parameter and summing values in another parameter
dfg = df.groupby(parameter_1)[parameter_2].sum()
dfg

In [ ]:
#Group the project by the number of elements in the group
dfcc = df.groupby(parameter_1)[parameter_2].count()
dfcc

In [ ]:
import matplotlib.pyplot as plt

colors_n1 = ['silver', 'steelblue', 'pink']
plot_n1 = dfcc.plot(kind='pie', figsize=(15, 15), autopct='%1.0f%%',colors=colors_n1, startangle=270)
plt.savefig(OUT / 'table_pie1.png', bbox_inches = 'tight') #Save as PNG in the output folder

In [ ]:
# Visualizing data from the Dataframe
dfcc.plot.barh(figsize=(15,30))

In [ ]:
# Let's form a graph with only non-zero values

parameter_2 = 'Volume'
x_label = 'Total ' + parameter_2 +', m3'

dfcc = dfcc.rename('Count')
df2 = pd.concat([dfg, dfcc], axis = 1)
df2 = df2[(df2.T != 0).any()]
df2 = df2.sort_values(parameter_1, ascending=False)
ax1 = df2.iloc[:20,:].plot(y=[parameter_2], kind="barh", fontsize = 16)
ax1.legend(fontsize = 15)
ax1.set_ylabel('Type', fontdict={'fontsize':20})
ax1.set_xlabel(x_label, fontdict={'fontsize':20})
plt.savefig(OUT / 'table_vol1.png', bbox_inches = 'tight') #Save as PNG in the output folder

In [ ]:
#Let's check if the parameters are filled in
# (once:  pip install missingno)

import missingno as msno
figm2 = msno.bar(df.iloc[: , 0:40])
fig_copym2 = figm2.get_figure()
fig_copym2.savefig(OUT / 'plot_pr2asm.png', bbox_inches = 'tight')